# Unit 1 Hands-On: LunarLander-v3 강화학습 실습

이 노트북은 **Hugging Face 딥 강화학습 강좌 Unit 1**의 실습입니다.  
PPO(Proximal Policy Optimization) 알고리즘을 이용해 `LunarLander-v3` 환경에서 에이전트를 훈련하고, Hugging Face Hub에 업로드하는 전 과정을 다룹니다.

---
## 목차
1. 결과 미리보기
2. 환경 설치
3. Google Drive 마운트
4. 가상 디스플레이 설정
5. 라이브러리 임포트
6. 환경 탐색
7. 모델 정의 및 훈련
8. 모델 평가
9. Hugging Face Hub에 업로드

---
## 1. 결과 미리보기

훈련이 완료된 PPO 에이전트가 LunarLander 환경에서 착륙하는 영상입니다.

In [ ]:
%%html
<video controls autoplay>
  <source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4">
</video>

---
## 2. 환경 설치

LunarLander 환경 실행에 필요한 시스템 패키지와 파이썬 패키지를 설치합니다.  
Colab 환경에서 한 번만 실행하면 됩니다.

In [ ]:
# Box2D 물리 엔진 의존성 및 빌드 도구 설치
!apt-get update && apt-get install -y \
    libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev \
    libportmidi-dev libfreetype6-dev \
    swig build-essential

In [ ]:
# 강좌 Unit 1에서 제공하는 requirements 파일로 파이썬 패키지 일괄 설치
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit1/requirements-unit1.txt

In [ ]:
# swig와 cmake 추가 설치 (Box2D 컴파일에 필요)
!apt install swig cmake

In [ ]:
# 가상 디스플레이 관련 패키지 설치
# Colab은 GUI가 없으므로 가상 디스플레이를 통해 렌더링을 처리합니다.
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay

---
## 3. Google Drive 마운트

Colab VM은 **세션 종료 시 모든 파일이 삭제**됩니다.  
Google Drive에 마운트하면 훈련 영상과 모델 파일이 영구적으로 보존됩니다.

```
저장 경로: Google Drive/LunarLander/
           ├── training_videos/        ← 훈련 중 단계별 영상
           └── ppo-LunarLander-v3.zip  ← 최종 모델
```

> 실행하면 Google 계정 인증 팝업이 뜹니다. 허용해주세요.

In [ ]:
from google.colab import drive
import os

# Google Drive 마운트 (/content/drive 에 연결)
drive.mount('/content/drive')

# ✏️ Drive 안에 저장할 폴더명을 원하는 대로 변경하세요.
DRIVE_BASE = "/content/drive/MyDrive/LunarLander"
VIDEO_DIR  = f"{DRIVE_BASE}/training_videos"
MODEL_DIR  = DRIVE_BASE

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"✅ Drive 마운트 완료!")
print(f"   영상 저장 경로 : {VIDEO_DIR}")
print(f"   모델 저장 경로 : {MODEL_DIR}")

---
## 4. 가상 디스플레이 설정

Colab(서버 환경)에는 모니터가 없기 때문에, 렌더링을 위한 **가상 화면(Virtual Display)**을 만들어야 합니다.

In [ ]:
from pyvirtualdisplay import Display

# 보이지 않는(headless) 1400x900 가상 디스플레이 시작
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
# Hugging Face와 Stable Baselines3 연동 패키지 설치
!pip install huggingface_sb3 stable_baselines3

# Box2D 기반 환경(LunarLander 등) 설치
!pip install "gymnasium[box2d]"

---
## 5. 라이브러리 임포트

| 라이브러리 | 역할 |
|---|---|
| `gymnasium` | 강화학습 환경 (OpenAI Gym의 후속) |
| `stable_baselines3` | PPO 등 강화학습 알고리즘 구현체 |
| `huggingface_sb3` | SB3 모델을 HF Hub에 업로드/다운로드 |
| `huggingface_hub` | HF Hub 로그인 등 인증 처리 |

In [ ]:
import imageio
import gymnasium as gym

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

from IPython.display import Video, display

---
## 6. 환경 탐색

훈련 전에 환경이 어떻게 생겼는지 먼저 살펴봅니다.

### 5-1. 랜덤 행동으로 환경 테스트

에이전트를 훈련하기 전에, **완전 랜덤한 행동**으로 20스텝을 실행해봅니다.

In [ ]:
env = gym.make("LunarLander-v3")
observation, info = env.reset()

for _ in range(20):
    action = env.action_space.sample()  # 랜덤 행동 선택
    print("Action taken:", action)

    # 행동 실행 → 다음 상태, 보상, 종료 여부 반환
    observation, reward, terminated, truncated, info = env.step(action)

    # 에피소드가 끝나면(착륙/충돌/시간초과) 환경 리셋
    if terminated or truncated:
        print("Environment is reset")
        observation, info = env.reset()

env.close()

### 5-2. 관측 공간(Observation Space) 확인

**관측 공간**: 에이전트가 환경으로부터 받는 정보의 형태입니다.  
LunarLander-v3의 관측값은 **8개의 연속 변수**로 구성됩니다:
- x, y 위치
- x, y 속도
- 각도 및 각속도
- 왼쪽/오른쪽 다리 접지 여부 (0 또는 1)

In [ ]:
env = gym.make("LunarLander-v3")
env.reset()

print("===== 관측 공간(Observation Space) =====")
print("Shape:", env.observation_space.shape)       # (8,) → 8개의 실수 값
print("Sample:", env.observation_space.sample())   # 랜덤 샘플 출력

### 5-3. 행동 공간(Action Space) 확인

**행동 공간**: 에이전트가 취할 수 있는 행동의 집합입니다.  
LunarLander-v3는 **이산(discrete) 행동 공간**을 가집니다:

| 행동 번호 | 의미 |
|:---:|---|
| 0 | 아무것도 하지 않음 |
| 1 | 왼쪽 엔진 점화 |
| 2 | 메인(하단) 엔진 점화 |
| 3 | 오른쪽 엔진 점화 |

In [ ]:
print("===== 행동 공간(Action Space) =====")
print("Action Space Size:", env.action_space.n)     # 4가지 행동
print("Sample Action:", env.action_space.sample())  # 랜덤 행동 샘플

---
## 7. 모델 정의 및 훈련

### 6-1. 벡터화 환경 생성

**벡터화 환경(Vectorized Environment)**: 여러 환경을 병렬로 동시에 실행하여 데이터 수집 효율을 높입니다.  
`n_envs=16`은 16개의 환경을 병렬 실행하여 훈련을 빠르게 합니다.

In [ ]:
# 16개의 LunarLander 환경을 병렬로 생성
env = make_vec_env('LunarLander-v3', n_envs=16)

### 6-2. PPO 모델 정의

**PPO(Proximal Policy Optimization)**: 안정적이고 효율적인 정책 기반 강화학습 알고리즘입니다.

| 하이퍼파라미터 | 값 | 설명 |
|---|---|---|
| `policy` | `'MlpPolicy'` | 다층 퍼셉트론(MLP) 기반 정책 네트워크 |
| `n_steps` | 1024 | 업데이트 전 각 환경에서 수집할 스텝 수 |
| `batch_size` | 64 | 미니배치 크기 |
| `n_epochs` | 4 | 데이터 재사용 횟수 |
| `gamma` | 0.999 | 미래 보상 할인율 (1에 가까울수록 장기 보상 중시) |
| `gae_lambda` | 0.98 | GAE 분산-편향 트레이드오프 조절 |
| `ent_coef` | 0.01 | 탐색을 장려하는 엔트로피 보너스 계수 |

In [ ]:
model = PPO(
    policy='MlpPolicy',
    env=env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    verbose=1  # 훈련 진행 상황 출력
)

### 6-3. 영상 저장 콜백 정의

> ⚠️ VSCode + Jupyter 환경에서 `render_mode="human"`(직접 창 띄우기)은  
> pygame과 Jupyter 커널이 충돌하여 **커널이 죽는 문제**가 있습니다.  
> 대신 `rgb_array`로 프레임을 캡처해 **mp4로 저장**하고 노트북 안에서 바로 재생합니다.

**동작 흐름**:
```
[훈련 환경 x16] ──학습──▶ [모델]
                               │
                  N스텝마다    ▼
          [rgb_array 환경] → 프레임 수집 → mp4 저장 → 노트북에서 재생
```

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `render_freq` | 30,000 | 몇 스텝마다 영상을 저장할지 |
| `n_eval_episodes` | 1 | 저장할 에피소드 수 |
| `video_dir` | `training_videos` | 영상 저장 폴더 |

In [ ]:
class VideoRenderCallback(BaseCallback):
    """
    학습 중 N스텝마다 에피소드를 mp4로 저장하는 콜백.
    - render_mode='human' 대신 'rgb_array'를 사용 → 커널 크래시 없음
    - 저장된 영상은 훈련 직후 아래 셀에서 바로 재생 가능
    - num_timesteps 기준으로 트리거 → n_envs가 몇이든 정확히 동작
    """

    def __init__(self, render_freq: int = 30_000, n_eval_episodes: int = 1,
                 video_dir: str = "training_videos"):
        super().__init__(verbose=0)
        self.render_freq = render_freq
        self.n_eval_episodes = n_eval_episodes
        self.video_dir = video_dir
        self.last_render_step = 0  # 마지막 저장 시점 추적
        os.makedirs(video_dir, exist_ok=True)

    def _on_step(self) -> bool:
        # 마지막 저장으로부터 render_freq 스텝 이상 지났을 때 실행
        # (n_calls 대신 num_timesteps 사용 → n_envs=16이어도 정확히 동작)
        if self.num_timesteps - self.last_render_step >= self.render_freq:
            self.last_render_step = self.num_timesteps
            print(f"\n🎬 [{self.num_timesteps:,} 스텝] 영상 저장 중...")

            try:
                # rgb_array 모드: 프레임을 numpy 배열로 받음 (창 안 띄움 → 커널 안전)
                render_env = gym.make("LunarLander-v3", render_mode="rgb_array")

                for ep in range(self.n_eval_episodes):
                    frames = []
                    obs, _ = render_env.reset()
                    done = False
                    total_reward = 0.0

                    while not done:
                        frames.append(render_env.render())  # 프레임 수집
                        action, _ = self.model.predict(obs, deterministic=True)
                        obs, reward, terminated, truncated, _ = render_env.step(action)
                        total_reward += reward
                        done = terminated or truncated

                    # mp4 파일로 저장 (파일명에 스텝 수 포함)
                    video_path = f"{self.video_dir}/step_{self.num_timesteps:07d}_ep{ep+1}.mp4"
                    imageio.mimsave(video_path, frames, fps=30)
                    print(f"   ✅ 저장 완료: {video_path}  (보상: {total_reward:.1f})")

                render_env.close()
                print(f"   다음 저장: {self.last_render_step + self.render_freq:,} 스텝")

            except Exception as e:
                print(f"   ⚠️ 영상 저장 실패: {e}")

        return True


# ✏️ render_freq 조절:
#   30_000 → 3만 스텝마다 저장 (자주 확인 가능, 영상 파일 많아짐)
#   50_000 → 5만 스텝마다 저장
render_callback = VideoRenderCallback(
    render_freq=30_000,
    n_eval_episodes=1,
    video_dir=VIDEO_DIR  # Google Drive 내 training_videos 폴더에 저장
)

### 6-4. imageio 설치

mp4 저장에 필요한 패키지입니다. 최초 1회만 실행하면 됩니다.

In [ ]:
!pip install imageio imageio-ffmpeg

### 6-5. 모델 훈련 및 저장

훈련을 시작합니다. 3만 스텝마다 `training_videos/` 폴더에 mp4가 자동 저장됩니다.

| 훈련 단계 | 예상 보상 | 영상에서 보이는 것 |
|---|---|---|
| 0 ~ 10만 스텝 | 음수 ~ 0 | 충돌, 제자리 맴돔 |
| 10만 ~ 50만 스텝 | 0 ~ 100 | 어설프게 착륙 시도 |
| 50만 ~ 100만 스텝 | 100 ~ 200+ | 안정적으로 착륙 |

In [ ]:
# 훈련 실행 (3만 스텝마다 training_videos/ 에 mp4 자동 저장)
model.learn(
    total_timesteps=1_000_000,
    callback=render_callback
)

# 모델 저장
model_name = "ppo-LunarLander-v3"
model_path = f"{MODEL_DIR}/{model_name}"
model.save(model_path)
print(f"\n✅ 훈련 완료! 모델이 '{model_path}.zip'으로 저장되었습니다.")


### 6-6. 저장된 영상 확인

훈련 중 저장된 영상 목록을 확인하고, 원하는 시점의 영상을 노트북 안에서 바로 재생합니다.

In [ ]:
import glob

# 저장된 영상 목록 출력
videos = sorted(glob.glob(f"{VIDEO_DIR}/*.mp4"))
print(f"총 {len(videos)}개의 영상이 저장되었습니다:\n")
for v in videos:
    print(" ", v)

In [ ]:
# ✏️ 보고 싶은 영상 파일명을 아래에 입력하세요.
# 예) 초반: videos[0], 중반: videos[len(videos)//2], 최종: videos[-1]

# 모든 영상을 순서대로 재생
for video_path in videos:
    step = video_path.split("step_")[1].split("_")[0]  # 파일명에서 스텝 수 추출
    print(f"\n📽️  {int(step):,} 스텝 시점")
    display(Video(video_path, embed=True, width=400))

---
## 8. 모델 평가

`evaluate_policy`를 사용해 10번의 에피소드 동안 평균 보상을 측정합니다.

- **좋은 점수 기준**: 평균 보상 200점 이상이면 성공적인 착륙으로 간주됩니다.
- `deterministic=True`: 평가 시에는 확률적 선택 대신 가장 좋은 행동만 선택합니다.

In [ ]:
# 평가용 환경 생성 (Monitor로 감싸서 보상 기록)
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))

# 10번 에피소드로 평균 보상 및 표준편차 계산
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print(f"평균 보상: {mean_reward:.2f} ± {std_reward:.2f}")

---
## 9. Hugging Face Hub에 모델 업로드

훈련한 모델을 커뮤니티와 공유하려면 Hugging Face Hub에 업로드합니다.

### 사전 준비
1. [Hugging Face 계정 생성](https://huggingface.co/join)
2. [쓰기(write) 권한의 토큰 발급](https://huggingface.co/settings/tokens)
3. 아래 셀에서 로그인

### 8-1. Hugging Face 로그인

In [ ]:
from huggingface_hub import login

# ✏️ 본인의 HF 토큰으로 교체하세요 (https://huggingface.co/settings/tokens)
# ⚠️ 토큰은 절대 GitHub 등 외부에 공개하지 마세요!
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxx")


### 8-2. 모델 업로드

`repo_id`를 **본인의 HF 사용자명/저장소명** 형식으로 변경하세요.  
예: `"홍길동/ppo-LunarLander-v3"`

In [ ]:
# ✏️ 아래 변수들을 본인 정보에 맞게 수정하세요.

env_id = "LunarLander-v3"           # 환경 이름
model_architecture = "PPO"           # 사용한 알고리즘
repo_id = "YOUR_HF_USERNAME/ppo-LunarLander-v3"  # ← 본인의 HF ID로 변경!
commit_message = "Upload PPO LunarLander-v3 trained agent"

# 평가용 환경 (영상 녹화를 위해 rgb_array 모드 사용)
eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])

# Hugging Face Hub에 모델 업로드
package_to_hub(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message
)

print(f"✅ 모델이 https://huggingface.co/{repo_id} 에 업로드되었습니다!")